# VisualCOT-Swap — Week 1 Pilot: LLaVA-1.5-7B

**Goal:** Prove the attribution pipeline works end-to-end on 20 samples.
**Deliverable:** `results/pilot/llava/pilot_summary.json` with `avg_steps >= 3`

**Runtime:** ~45 min on a strong GPU runtime; use the fallback ladder if hooks fail or compute is tight.

### What this notebook does:
1. Installs dependencies
2. Locates the repo automatically, even if it is not the current working directory
3. Downloads ScienceQA if needed
4. Builds a mini 20-triplet benchmark
5. Runs GFS attribution on LLaVA-1.5-7B (20 samples)
6. Shows qualitative heatmaps for 3 samples
7. Saves the pilot summary and local artifacts

## Cell 1: Install dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root() -> Path:
    env_root = os.environ.get('COHERENCE_TAX_ROOT') or os.environ.get('REPO_DIR')
    if env_root:
        env_path = Path(env_root).expanduser().resolve()
        if (env_path / 'requirements.txt').exists() and (env_path / 'scripts').exists():
            return env_path

    search_roots = [Path.cwd(), Path.home(), Path('/workspace'), Path('/root'), Path('/mnt/data'), Path('/tmp'), Path('/content')]
    seen = set()
    for root in search_roots:
        if not root.exists():
            continue
        for candidate in [root, *root.parents]:
            if candidate in seen:
                continue
            seen.add(candidate)
            if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
                return candidate.resolve()
            try:
                for marker in candidate.glob('**/requirements.txt'):
                    repo = marker.parent
                    if (repo / 'scripts').exists() and (repo / 'src').exists():
                        return repo.resolve()
            except Exception:
                pass

    raise FileNotFoundError(
        'Could not locate repo root automatically. Set COHERENCE_TAX_ROOT or REPO_DIR to the repo path.'
    )


def run_pip_install(args, label, allow_failure=False):
    print(f'Installing {label}: {args}')
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args])
    if result.returncode != 0:
        message = f'{label} install failed with return code {result.returncode}'
        if allow_failure:
            print('WARNING:', message)
            return False
        raise RuntimeError(message)
    return True


REPO_DIR = str(find_repo_root())
requirements_path = Path(REPO_DIR) / 'requirements.txt'
base_packages = []
for raw_line in requirements_path.read_text().splitlines():
    line = raw_line.strip()
    if not line or line.startswith('#'):
        continue
    if line.startswith('torch==') or line.startswith('bitsandbytes=='):
        continue
    base_packages.append(line)

run_pip_install(['--upgrade', 'pip', 'setuptools', 'wheel'], 'packaging tools', allow_failure=True)
run_pip_install(base_packages, 'core requirements')
run_pip_install(['bitsandbytes==0.42.0'], 'bitsandbytes pinned', allow_failure=True)
run_pip_install(['bitsandbytes'], 'bitsandbytes latest fallback', allow_failure=True)

try:
    import torch
    print(f'PyTorch available: {torch.__version__}')
except Exception as exc:
    print(f'PyTorch not importable yet: {exc}')

try:
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=True)
except subprocess.CalledProcessError as exc:
    print(f'WARNING: spaCy model download failed: {exc}')

print('Install done.')

## Cell 2: Initialize local workspace

In [ ]:
import os
from pathlib import Path

def find_repo_root() -> Path:
    env_root = os.environ.get('COHERENCE_TAX_ROOT') or os.environ.get('REPO_DIR')
    if env_root:
        env_path = Path(env_root).expanduser().resolve()
        if (env_path / 'requirements.txt').exists() and (env_path / 'scripts').exists():
            return env_path

    search_roots = [Path.cwd(), Path.home(), Path('/workspace'), Path('/root'), Path('/mnt/data'), Path('/tmp'), Path('/content')]
    seen = set()
    for root in search_roots:
        if not root.exists():
            continue
        for candidate in [root, *root.parents]:
            if candidate in seen:
                continue
            seen.add(candidate)
            if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
                return candidate.resolve()
            try:
                for marker in candidate.glob('**/requirements.txt'):
                    repo = marker.parent
                    if (repo / 'scripts').exists() and (repo / 'src').exists():
                        return repo.resolve()
            except Exception:
                pass

    raise FileNotFoundError(
        'Could not locate repo root automatically. Set COHERENCE_TAX_ROOT or REPO_DIR to the repo path.'
    )

REPO_DIR = str(find_repo_root())
os.chdir(REPO_DIR)
RESULTS_DIR = f'{REPO_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Repo root: {REPO_DIR}')
print(f'Results dir: {RESULTS_DIR}')

## Cell 3: Confirm local repo access

In [ ]:
import os
print(f'Working dir: {os.getcwd()}')
print('Using the local notebook repository directly.')

## Cell 4: Download ScienceQA

In [ ]:
import os
import subprocess

LOCAL_DATA = f'{REPO_DIR}/data'
scienceqa_marker = f'{LOCAL_DATA}/scienceqa/metadata.json'

if os.path.exists(scienceqa_marker):
    print('ScienceQA already present locally; skipping download.')
else:
    print('Downloading ScienceQA...')
    subprocess.run(['python', 'scripts/prepare_data.py', '--output', LOCAL_DATA,
                    '--max_scienceqa', '5000', '--max_gqa', '0'], check=True)
    print('Local data download complete.')

## Cell 5: Build 20-triplet benchmark (pilot)

In [ ]:
subprocess.run([
    'python', 'scripts/build_benchmark.py',
    '--n', '20',
    '--output', f'{LOCAL_DATA}/swap_pairs',
    '--dry_run'
])
print('Benchmark built.')

## Cell 6: Run GFS pilot on LLaVA-1.5-7B

This is the core experiment. Runs 20 samples.
Expected time: ~30-40 min on A100 for the attribution run itself, or ~60-90 min end-to-end including install/download/checkpoint sync.

**Fallback ladder:**
- Try Grad-CAM first.
- If hooks fail or heatmaps look uniform, rerun with rollout attribution.
- If compute time is tight or memory errors persist, rerun with 10 samples and `--no_swap` as the last resort.

**What to watch for:**
- `avg_steps >= 3` in pilot_summary.json ✓
- Heatmaps are non-uniform (not all gray) ✓
- GFS values vary between steps ✓

In [ ]:
PILOT_OUT = f'{REPO_DIR}/results/pilot'
os.makedirs(PILOT_OUT, exist_ok=True)

import json
import subprocess
import time


def read_summary():
    summary_path = f'{PILOT_OUT}/llava/pilot_summary.json'
    if os.path.exists(summary_path):
        with open(summary_path) as f:
            return json.load(f)
    return None


def run_once(attribution, samples, no_swap=False, label='run'):
    cmd = [
        'python', 'scripts/run_attribution.py',
        '--model', 'llava',
        '--samples', str(samples),
        '--output', PILOT_OUT,
        '--pilot',
        '--attribution', attribution,
    ]
    if no_swap:
        cmd.append('--no_swap')

    print(f'\n=== {label} ===')
    print('Command:', ' '.join(cmd))
    started = time.time()
    proc = subprocess.run(cmd, check=False)
    elapsed_min = (time.time() - started) / 60.0
    print(f'{label} finished in {elapsed_min:.1f} min with return code {proc.returncode}')
    return proc.returncode, elapsed_min


fallback_plan = [
    {'label': 'primary-gradcam', 'attribution': 'gradcam', 'samples': 20, 'no_swap': False},
    {'label': 'fallback-rollout', 'attribution': 'rollout', 'samples': 20, 'no_swap': False},
    {'label': 'fallback-small-noswap', 'attribution': 'rollout', 'samples': 10, 'no_swap': True},
]

run_times = []
for item in fallback_plan:
    rc, elapsed_min = run_once(
        attribution=item['attribution'],
        samples=item['samples'],
        no_swap=item['no_swap'],
        label=item['label'],
    )
    run_times.append(elapsed_min)
    summary = read_summary()
    if summary:
        avg_steps = summary.get('avg_steps', 0)
        print(f'Current avg_steps: {avg_steps}')
        if avg_steps >= 3 and rc == 0:
            print('Pilot deliverable reached; stopping fallback ladder.')
            break
        if avg_steps < 3:
            print('avg_steps < 3, moving to next fallback.')
    else:
        print('pilot_summary.json not found yet, moving to next fallback if available.')

print(f'\nTotal elapsed across attempts: {sum(run_times):.1f} min')
summary = read_summary()
if summary:
    print('Final pilot summary path:', f'{PILOT_OUT}/llava/pilot_summary.json')
    print(json.dumps(summary, indent=2))

## Cell 7: Load and display pilot summary

In [ ]:
import json, glob

summary_path = f'{PILOT_OUT}/llava/pilot_summary.json'
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print('=== PILOT SUMMARY ===')
    print(json.dumps(summary, indent=2))
    if summary.get('avg_steps', 0) >= 3:
        print('\n✓ WEEK 1 DELIVERABLE MET: avg_steps >= 3')
    else:
        print('\n✗ avg_steps < 3 — check CoT prompting template')
else:
    print('Summary not found. Check run_attribution.py output above.')

## Cell 8: GFS decay plot for pilot

In [ ]:
import json, glob
import numpy as np
import matplotlib.pyplot as plt

result_files = glob.glob(f'{PILOT_OUT}/llava/*.json')
result_files = [f for f in result_files if 'checkpoint' not in f and 'summary' not in f]

results = []
for f in result_files:
    with open(f) as fh:
        results.append(json.load(fh))

MAX_STEPS = 6
by_step = [[] for _ in range(MAX_STEPS)]
for r in results:
    for k, g in enumerate(r.get('gfs_per_step', [])[:MAX_STEPS]):
        if g is not None:
            by_step[k].append(g)

means = [np.mean(s) if s else np.nan for s in by_step]
sems  = [np.std(s)/np.sqrt(len(s)) if len(s) > 1 else 0 for s in by_step]
valid = [i for i, m in enumerate(means) if not np.isnan(m)]

fig, ax = plt.subplots(figsize=(5, 3.5))
x = np.array(valid) + 1
y = np.array([means[i] for i in valid])
e = np.array([sems[i] for i in valid])
ax.plot(x, y, 'o-', color='#534AB7', label='LLaVA-1.5-7B (pilot)')
ax.fill_between(x, y-e, y+e, color='#534AB7', alpha=0.2)
ax.set_xlabel('Reasoning step k'); ax.set_ylabel('Mean GFS')
ax.set_ylim(0, 1); ax.legend(frameon=False)
ax.set_title(f'GFS Decay — Pilot (n={len(results)} samples)')
plt.tight_layout()
plt.savefig(f'{PILOT_OUT}/llava/pilot_gfs_decay.png', dpi=150)
plt.show()
print(f'Steps with data: {[i+1 for i in valid]}')
print(f'Mean GFS by step: {[f"{m:.3f}" if not np.isnan(m) else "—" for m in means]}')

## Cell 9: Qualitative check — heatmap interpretation

**Check manually:**
- Step 1 heatmap should focus on salient image region
- Step 3-4 heatmap should be more diffuse
- If all heatmaps are uniform → Grad-CAM hooks failed → switch to rollout

In [ ]:
# Show first result's CoT steps and GFS values side-by-side
if results:
    r = results[0]
    print('=== SAMPLE RESULT ===')
    print(f'Question: {r["question"]}')
    print(f'N steps: {r["n_steps"]}')
    print(f'GFS per step: {r["gfs_per_step"]}')
    print(f'Decay slope: {r.get("gfs_decay_slope")}')
    print(f'\nCoT text:\n{r["cot_text"][:800]}')

## Cell 10: Local artifact summary

This final cell lists the files produced by the pilot run.

In [ ]:
import json
from pathlib import Path

results_path = Path(f'{REPO_DIR}/results')
print('Local results tree:')
for path in sorted(results_path.rglob('*')):
    if path.is_file():
        print(path.relative_to(REPO_DIR))
print('Notebook finished. Results remain in the local repo under results/.')